In [2]:
import pandas as pd

reviews_raw = pd.read_csv("student_data/reviews.csv")
products_raw = pd.read_csv("student_data/products.csv")
promotions_raw = pd.read_csv("student_data/promotions.csv")
returns_raw = pd.read_csv("student_data/returns.csv")

print("Reviews:", reviews_raw.shape)
print("Products:", products_raw.shape)
print("Promotions:", promotions_raw.shape)
print("Returns:", returns_raw.shape)

Reviews: (113551, 7)
Products: (2412, 8)
Promotions: (50, 10)
Returns: (39939, 7)


In [3]:
reviews = reviews_raw[
    [
        "review_id",
        "order_id",
        "product_id",
        "customer_id",
        "review_date",
        "rating",
        "review_title"
    ]
].copy()

# Clean text
for col in ["review_id", "review_title"]:
    reviews[col] = reviews[col].astype("string").str.strip()

# Clean numeric columns
for col in ["order_id", "product_id", "customer_id", "rating"]:
    reviews[col] = pd.to_numeric(
        reviews[col],
        errors="coerce"
    ).astype("Int64")

# Clean date
reviews["review_date"] = pd.to_datetime(
    reviews["review_date"],
    errors="coerce"
)

# Bỏ các dòng không có khóa chính
reviews = reviews.dropna(
    subset=["review_id"]
)

# Mỗi review_id chỉ xuất hiện một lần
reviews = reviews.drop_duplicates(
    subset=["review_id"]
).reset_index(drop=True)

reviews.head()

,review_id,order_id,product_id,customer_id,review_date,rating,review_title
0,REV-0000001,1,2400,58578,2012-07-24,5,Highly recommend
1,REV-0000002,3,396,58811,2012-08-03,5,Very satisfied
2,REV-0000003,10,1431,49101,2012-07-23,5,Great quality
3,REV-0000005,16,1668,41028,2012-08-05,5,Great quality
4,REV-0000006,17,2352,42030,2012-07-17,4,Good overall


In [4]:
# Check REVIEWS
print(reviews.info())
print("\nMissing values:")
print(reviews.isnull().sum())
print("\nReview ID trùng:", reviews["review_id"].duplicated().sum())
print("Rating ngoài khoảng 1-5:", ((reviews["rating"] < 1) | (reviews["rating"] > 5)).sum())

<class 'pandas.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   review_id     113551 non-null  string        
 1   order_id      113551 non-null  Int64         
 2   product_id    113551 non-null  Int64         
 3   customer_id   113551 non-null  Int64         
 4   review_date   113551 non-null  datetime64[us]
 5   rating        113551 non-null  Int64         
 6   review_title  113551 non-null  string        
dtypes: Int64(4), datetime64[us](1), string(2)
memory usage: 6.5 MB
None

Missing values:
review_id       0
order_id        0
product_id      0
customer_id     0
review_date     0
rating          0
review_title    0
dtype: int64

Review ID trùng: 0
Rating ngoài khoảng 1-5: 0


In [5]:
products = products_raw[
    [
        "product_id",
        "product_name",
        "category",
        "segment",
        "size",
        "color",
        "price",
        "cogs"
    ]
].copy()

# Clean text
for col in ["product_name", "category", "segment", "size", "color"]:
    products[col] = products[col].astype("string").str.strip()

# Clean numeric columns
products["product_id"] = pd.to_numeric(
    products["product_id"],
    errors="coerce"
).astype("Int64")

for col in ["price", "cogs"]:
    products[col] = pd.to_numeric(
        products[col],
        errors="coerce"
    )

products = products.dropna(
    subset=["product_id"]
)

products = products.drop_duplicates(
    subset=["product_id"]
).reset_index(drop=True)

products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [6]:
# Check PRODUCTS
print(products.info())
print("\nMissing values:")
print(products.isnull().sum())
print("\nProduct ID trùng:", products["product_id"].duplicated().sum())
print("Price <= 0:", (products["price"] <= 0).sum())
print("COGS <= 0:", (products["cogs"] <= 0).sum())
print("COGS > Price:", (products["cogs"] > products["price"]).sum())

<class 'pandas.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   Int64  
 1   product_name  2412 non-null   string 
 2   category      2412 non-null   string 
 3   segment       2412 non-null   string 
 4   size          2412 non-null   string 
 5   color         2412 non-null   string 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: Int64(1), float64(2), string(5)
memory usage: 153.2 KB
None

Missing values:
product_id      0
product_name    0
category        0
segment         0
size            0
color           0
price           0
cogs            0
dtype: int64

Product ID trùng: 0
Price <= 0: 0
COGS <= 0: 0
COGS > Price: 0


In [7]:
promotions = promotions_raw[
    [
        "promo_id",
        "promo_name",
        "promo_type",
        "discount_value",
        "start_date",
        "end_date",
        "applicable_category",
        "promo_channel",
        "stackable_flag",
        "min_order_value"
    ]
].copy()

# Clean text
for col in [
    "promo_id",
    "promo_name",
    "promo_type",
    "applicable_category",
    "promo_channel"
]:
    promotions[col] = promotions[col].astype("string").str.strip()

# Clean numeric columns
for col in ["discount_value", "stackable_flag", "min_order_value"]:
    promotions[col] = pd.to_numeric(
        promotions[col],
        errors="coerce"
    )

promotions["stackable_flag"] = promotions["stackable_flag"].astype("Int64")
promotions["min_order_value"] = promotions["min_order_value"].astype("Int64")

# Clean dates
for col in ["start_date", "end_date"]:
    promotions[col] = pd.to_datetime(
        promotions[col],
        errors="coerce"
    )

promotions = promotions.dropna(
    subset=["promo_id"]
)

promotions = promotions.drop_duplicates(
    subset=["promo_id"]
).reset_index(drop=True)

promotions.head()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,<NA>,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,<NA>,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,<NA>,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,<NA>,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


In [8]:
# Check PROMOTIONS
print(promotions.info())
print("\nMissing values:")
print(promotions.isnull().sum())
print("\nPromo ID trùng:", promotions["promo_id"].duplicated().sum())
print("Start date > End date:", (promotions["start_date"] > promotions["end_date"]).sum())
print("Discount <= 0:", (promotions["discount_value"] <= 0).sum())
print("Stackable flag khác 0/1:", (~promotions["stackable_flag"].isin([0, 1])).sum())

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     string        
 1   promo_name           50 non-null     string        
 2   promo_type           50 non-null     string        
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[us]
 5   end_date             50 non-null     datetime64[us]
 6   applicable_category  10 non-null     string        
 7   promo_channel        50 non-null     string        
 8   stackable_flag       50 non-null     Int64         
 9   min_order_value      50 non-null     Int64         
dtypes: Int64(2), datetime64[us](2), float64(1), string(5)
memory usage: 4.1 KB
None

Missing values:
promo_id                0
promo_name              0
promo_type              0
discount_value          0
start_date        

In [9]:
returns = returns_raw[
    [
        "return_id",
        "order_id",
        "product_id",
        "return_date",
        "return_reason",
        "return_quantity",
        "refund_amount"
    ]
].copy()

# Clean text
for col in ["return_id", "return_reason"]:
    returns[col] = returns[col].astype("string").str.strip()

# Clean numeric columns
for col in ["order_id", "product_id", "return_quantity"]:
    returns[col] = pd.to_numeric(
        returns[col],
        errors="coerce"
    ).astype("Int64")

returns["refund_amount"] = pd.to_numeric(
    returns["refund_amount"],
    errors="coerce"
)

# Clean date
returns["return_date"] = pd.to_datetime(
    returns["return_date"],
    errors="coerce"
)

returns = returns.dropna(
    subset=["return_id"]
)

returns = returns.drop_duplicates(
    subset=["return_id"]
).reset_index(drop=True)

returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [10]:
# Check RETURNS
print(returns.info())
print("\nMissing values:")
print(returns.isnull().sum())
print("\nReturn ID trùng:", returns["return_id"].duplicated().sum())
print("Return quantity <= 0:", (returns["return_quantity"] <= 0).sum())
print("Refund amount <= 0:", (returns["refund_amount"] <= 0).sum())

<class 'pandas.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   return_id        39939 non-null  string        
 1   order_id         39939 non-null  Int64         
 2   product_id       39939 non-null  Int64         
 3   return_date      39939 non-null  datetime64[us]
 4   return_reason    39939 non-null  string        
 5   return_quantity  39939 non-null  Int64         
 6   refund_amount    39939 non-null  float64       
dtypes: Int64(3), datetime64[us](1), float64(1), string(2)
memory usage: 2.2 MB
None

Missing values:
return_id          0
order_id           0
product_id         0
return_date        0
return_reason      0
return_quantity    0
refund_amount      0
dtype: int64

Return ID trùng: 0
Return quantity <= 0: 0
Refund amount <= 0: 0


In [11]:
print("===== FINAL CHECK =====")
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Promotions:", promotions.shape)
print("Returns:", returns.shape)

print("\nNull reviews:", reviews.isnull().sum().sum())
print("Null products:", products.isnull().sum().sum())
print("Null promotions:", promotions.isnull().sum().sum())
print("Null returns:", returns.isnull().sum().sum())

===== FINAL CHECK =====
Reviews: (113551, 7)
Products: (2412, 8)
Promotions: (50, 10)
Returns: (39939, 7)

Null reviews: 0
Null products: 0
Null promotions: 40
Null returns: 0
